## Notebook 05: Hydrogen Case Study Results

This notebook analyses and visualises the LCA results from notebook 04. It applies stochastic multicriteria acceptability analysis (SMAA) to the social indicators and compares the results with environmental ReCiPe endpoint totals.

**What it does:**
1. Reads the per-(technology, country) CSV files from notebook 04.
2. Normalises social indicators within each country (min-max scaling).
3. Runs SMAA with Dirichlet-sampled weights for each stakeholder focus regime.
4. Produces a 3x3 choropleth map grid (6 stakeholder panels + 3 ReCiPe endpoint panels) showing the best technology per country.
5. Produces a robustness plot comparing a strong and weak country.

**Inputs:**
- CSV result files from notebook 04 (`Data/LCA_contribution_data/Nodes/`).

**Outputs:**
- `figure1_stakeholder_recipe_grid.svg`: 3x3 map grid (174 mm wide).
- `figure2_robustness_demo.svg`: acceptability and score margin comparison.

In [ ]:
# --- Imports ---
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional, Sequence
import re
import zlib
import json
from datetime import datetime
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from matplotlib.patches import Patch, Rectangle
import yaml
from IPython.display import display

# --- Reproducibility / display ---
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

def log(event: str, **kwargs):
    """Small logger (kept minimal for a standalone notebook)."""
    # print(event, kwargs)  # uncomment if you want logs
    pass

# =============================================================================
# USER INPUTS (EDIT THESE)
# =============================================================================
RAW_DIR = Path("../Data/LCA_contribution_data/Nodes")
FIG_SAVE_PATH: Optional[Path] = Path("figure1_stakeholder_recipe_grid.svg")
VARIANT = "EXP"  # "EXP" or "LIN" (we run EXP only here)

# Optional: cache basemap locally for reproducibility and speed
WORLD_FILE: Optional[Path] = None  # if you have a local Natural Earth admin_0 layer
WORLD_CACHE_DIR = Path(".cache_natural_earth")

# Optional: write a small run manifest (recommended for paper SI)
MANIFEST_PATH: Optional[Path] = None  # e.g., Path("hydrogen_case_study_manifest.json")
FIG2_SAVE_PATH: Optional[Path] = Path("figure2_robustness_demo.svg")

# Figure geometry
DPI = 600
COL_GAP_MM = 1.0
ROW_GAP_MM = 6.0
TOP_MARGIN_MM = 20.0
BOTTOM_MARGIN_MM = 31.0
CROP_SOUTH_LAT = -60.0

# Uncertainty cue on maps (stakeholder panels): hatch if best acceptability < threshold
ACCEPTABILITY_HATCH_THRESHOLD = 0.50

# Normalization: treat indicator as non-discriminating within a country if range < EPS
NORM_EPS = 1e-12

# Sanity checks
assert RAW_DIR.exists(), f"RAW_DIR does not exist: {RAW_DIR.resolve()}"

# =============================================================================
# Robust text canonicalization (prevents silent config mismatches)
# =============================================================================
def _canon_text(x: Any) -> str:
    return str(x).strip()

def _canon_key(x: Any) -> str:
    return _canon_text(x).casefold()

# Standardize stakeholder display names (so YAML / data capitalization differences don’t break logic)
STAKEHOLDER_DISPLAY: Dict[str, str] = {
    "children": "Children",
    "consumer": "Consumer",
    "local community": "Local Community",
    "local community ": "Local Community",
    "society": "Society",
    "value chain actors": "Value Chain Actors",
    "workers": "Workers",
}

def canon_stakeholder(x: Any) -> str:
    k = _canon_key(x)
    return STAKEHOLDER_DISPLAY.get(k, _canon_text(x))

def canon_subgroup(x: Any) -> str:
    # we keep subgroup text, but comparisons will use casefold
    return _canon_text(x)

# =============================================================================
# Technology canonicalization (for consistent checks + labels + figure 2)
# =============================================================================
TECH_LABELS: Dict[str, str] = {
    "biomass_gasification": "Biomass gasification",
    "electrolysis": "Electrolysis",
    "smr_biomethane": "SMR (biomethane)",
    "smr_natural_gas": "SMR (natural gas)",
    "coal_gasification": "Coal gasification",
}

ALIASES: Dict[str, str] = {
    "pem_electrolysis__green": "electrolysis",
    "steam_methane_reforming": "smr_natural_gas",
    "steam_methane_reforming_of_biomethane": "smr_biomethane",
    "biomass_gasification": "biomass_gasification",
    "coal_gasification": "coal_gasification",
}

def normalize_tech_code(raw: Any) -> str:
    s = _canon_text(raw)
    return ALIASES.get(s, s)

def tech_clean_label(tech_code: Any) -> str:
    canon = normalize_tech_code(tech_code)
    return TECH_LABELS.get(canon, str(tech_code))

def canonicalize_techs_inplace(df: pd.DataFrame, col: str = "tech") -> pd.DataFrame:
    out = df.copy()
    out[col] = out[col].apply(normalize_tech_code)
    if out.duplicated(subset=["country", col]).any():
        dups = out.loc[out.duplicated(subset=["country", col], keep=False), ["country", col]].sort_values(["country", col])
        raise ValueError(
            "Canonicalizing technology codes created duplicate (country, tech) rows. "
            "This indicates the same tech is present under multiple aliases. Examples:\n"
            + dups.head(30).to_string(index=False)
        )
    return out

def report_tech_coverage(df: pd.DataFrame, expected_techs: Optional[Sequence[str]] = None, max_rows: int = 30) -> None:
    """
    Print a coverage report: whether every country has every expected technology.
    Does NOT raise; purely informative.
    """
    expected = list(expected_techs) if expected_techs is not None else sorted(df["tech"].unique())
    expected_set = set(expected)

    by_country = []
    for c, g in df.groupby("country", sort=True):
        present = set(g["tech"].astype(str))
        missing = sorted(expected_set - present)
        extra = sorted(present - expected_set)
        if missing or extra:
            by_country.append({"country": c, "missing_techs": ", ".join(missing), "extra_techs": ", ".join(extra)})

    print("\n=== Technology coverage check ===")
    print(f"Expected technologies (n={len(expected)}): {', '.join(expected)}")
    print(f"Countries: {df['country'].nunique()} | (country, tech) rows: {len(df)}")

    if not by_country:
        print("✔ All countries contain the full expected technology set.")
        return

    rep = pd.DataFrame(by_country)
    n_incomplete = (rep["missing_techs"].astype(str).str.len() > 0).sum()
    print(f"⚠ {n_incomplete}/{df['country'].nunique()} countries are missing at least one expected technology.")
    display(rep.head(max_rows))

# =============================================================================
# CONFIG (embedded YAML)
# =============================================================================
PROD_YAML = r"""filters:
  domain_by_stakeholder:
    "ReCiPe 2016 v1.03, endpoint (H) regionalized": env
    "ReCiPe 2016 v1.03, midpoint (H) regionalized": env
    "EF v3.1 regionalized": env

    Children: soc
    Consumer: soc
    "Local community": soc
    Society: soc
    "Value Chain Actors": soc
    Workers: soc

  rows:
    exclude_countries: []
    exclude_techs:
      - gaseous__petroleum_refinery_operation

  columns:
    exclude_stakeholders:
      - "ReCiPe 2016 v1.03, endpoint (H) regionalized"
      - "ReCiPe 2016 v1.03, midpoint (H) regionalized"
      - "EF v3.1 regionalized"

    exclude_subgroups:
      - { stakeholder: "Local Community", subgroup: "Environmental footprints" }
      - { stakeholder: "Local Community", subgroup: "GHG footprints" }
      - { stakeholder: "Society", subgroup: "Education and upskilling opportunities" }
      - { stakeholder: "Society", subgroup: "Gender equality and empowerment" }
      - { stakeholder: "Society", subgroup: "Censorship and oppression" }
      - { stakeholder: "Society", subgroup: "Nature" }
      - { stakeholder: "Society", subgroup: "Governance" }
      - { stakeholder: "Society", subgroup: "Health and safety" }

    exclude_indicators: []
    exclude_regex: []
    whitelist_indicators: []

  meta_overrides:
    direction_good:
      - "Workers | Social benefits, legal issues | Paid maternity leave"
      - "Society | Contribution to economic development | Embodied value added total"
      - "Society | Contribution to economic development | Contribution to economic development"
      - "Society | Gender equality and empowerment | Female with account at a financial institution"
    domain_env: []
    domain_soc: []

smaa:
  n_draws: 50000
  seed: 42

  # NOTE: alpha_stakeholder is not used in this stakeholder-focus design,
  # because each regime assigns 100% weight within the focused stakeholder.
  alpha_stakeholder:
    default: 1.0
    Workers: 1.0
    Society: 1.0
    "Local Community": 1.0
    Children: 1.0
    Consumer: 1.0
    "Value Chain Actors": 1.0

  alpha_subgroup: 1.0
  alpha_indicators: 1.0

  deterministic_equal_stakeholders: false
  deterministic_equal_subgroups: false
  deterministic_equal_within_subgroup: false
"""
CFG = yaml.safe_load(PROD_YAML)

CFG.setdefault("smaa", {})
CFG["smaa"]["n_draws"] = int(CFG["smaa"].get("n_draws", 50000))
CFG["smaa"]["seed"] = int(CFG["smaa"].get("seed", 42))

CFG.setdefault("filters", {})
CFG["filters"].setdefault("rows", {})
CFG["filters"]["rows"].setdefault("exclude_countries", [])
CFG["filters"]["rows"].setdefault("exclude_techs", [])

# =============================================================================
# 1) Data ingestion: build a (country, tech) × indicators score matrix
# =============================================================================
META_COLS = {"index", "name", "reference product", "location", "unit", "database"}

def _parse_tech_country(path: Path) -> Tuple[str, str]:
    """
    Extract (country, tech) from filenames like:
        hydrogen_production__<TECH_PARTS>_<COUNTRY>.csv
    where COUNTRY is the last underscore-separated token (2–3 letters).
    """
    stem = path.stem
    m = re.search(r"hydrogen_production__(.+)", stem)
    if not m:
        raise ValueError(f"Cannot parse tech/country from filename stem: {stem}")
    tail = m.group(1)
    parts = tail.split("_")
    if len(parts) < 2:
        raise ValueError(f"Not enough '_' parts to parse tech/country from: {stem}")
    country = parts[-1]
    if not (country.isalpha() and 2 <= len(country) <= 3):
        raise ValueError(f"Last token does not look like a country code: {country} in {stem}")
    tech = "_".join(parts[:-1])
    return country, tech

def _read_score_row(fp: Path) -> Optional[pd.Series]:
    """Read a single CSV node file and return the row labeled 'Score' as a numeric Series."""
    df = pd.read_csv(fp, low_memory=False)
    first_col = df.columns[0]
    score_row = df.loc[df[first_col].astype(str).str.strip().str.lower() == "score"]
    if score_row.empty:
        return None
    indicator_cols = [c for c in df.columns if _canon_key(c) not in META_COLS]
    s = score_row.iloc[0][indicator_cols]
    return pd.to_numeric(s, errors="coerce")

def build_score_matrix(raw_dir: str | Path) -> pd.DataFrame:
    raw_dir = Path(raw_dir)
    files = sorted(raw_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {raw_dir}")
    rows = []
    skipped = 0
    for fp in files:
        try:
            country, tech = _parse_tech_country(fp)
        except ValueError as e:
            skipped += 1
            log("filename_skip", file=fp.name, reason=str(e))
            continue
        s = _read_score_row(fp)
        if s is None:
            skipped += 1
            log("no_score_row", file=fp.name)
            continue
        row = {"country": country, "tech": tech}
        for c, v in s.items():
            row[str(c)] = float(v) if pd.notna(v) else np.nan
        rows.append(row)

    if not rows:
        raise RuntimeError("No valid 'Score' rows parsed. Check file structure / naming.")

    df = pd.DataFrame(rows)

    if df.duplicated(subset=["country", "tech"]).any():
        dups = df.loc[df.duplicated(subset=["country", "tech"], keep=False), ["country", "tech"]]
        raise ValueError(
            "Duplicate (country, tech) detected after ingestion. "
            "This should not happen. Examples:\n" + dups.head(20).to_string(index=False)
        )

    log("nodes_loaded", rows=len(df), cols=len(df.columns), skipped=skipped)
    return df

# =============================================================================
# Country code standardization: require ISO3 (convert ISO2 via pycountry if needed)
# =============================================================================
try:
    import pycountry  # type: ignore
except Exception:
    pycountry = None

def standardize_country_to_iso3(df: pd.DataFrame, col: str = "country") -> pd.DataFrame:
    """
    Ensure df[col] contains ISO3 codes (AAA).
    - If ISO2 exists, convert using pycountry (required).
    - If anything cannot be converted, raise.
    """
    out = df.copy()
    codes = out[col].astype(str).str.strip()

    bad_shape = codes.loc[~codes.str.match(r"^[A-Za-z]{2,3}$")]
    if len(bad_shape) > 0:
        raise ValueError(
            f"Country codes must be 2–3 letters. Found invalid examples: {bad_shape.head(10).tolist()}"
        )

    need_convert = codes.str.len().eq(2)
    if need_convert.any():
        if pycountry is None:
            raise RuntimeError(
                "Found ISO2 country codes but `pycountry` is not installed. "
                "Install pycountry or provide ISO3 codes in filenames."
            )

        def to_iso3(x: str) -> str:
            x = x.upper()
            if len(x) == 3:
                return x
            c = pycountry.countries.get(alpha_2=x)
            if not c:
                raise ValueError(f"Could not convert ISO2 to ISO3: {x}")
            return c.alpha_3

        out[col] = codes.apply(to_iso3)

    final = out[col].astype(str).str.strip()
    bad3 = final.loc[~final.str.match(r"^[A-Za-z]{3}$")]
    if len(bad3) > 0:
        raise ValueError(f"Expected ISO3 country codes after standardization; bad examples: {bad3.head(10).tolist()}")

    out[col] = final.str.upper()
    return out

# =============================================================================
# 2) Filtering + metadata (domain / stakeholder / subgroup / direction)
# =============================================================================
KEYS = {"country", "tech"}
_GENDER_TOKENS = {"male", "female", "m", "f", "total"}

def _col_list(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c not in KEYS]

def _split_gender_suffix(s: str) -> Tuple[str, Optional[str]]:
    parts = [p.strip() for p in str(s).split("|")]
    if not parts:
        return s, None
    last = parts[-1].strip()
    if last.lower() in _GENDER_TOKENS:
        base = " | ".join(parts[:-1]).strip()
        return base, last.lower()
    m = re.match(r"^(.*?)(?:\s+)(male|female|m|f|total)$", str(s).strip(), flags=re.IGNORECASE)
    if m:
        base = m.group(1).strip()
        gender = m.group(2).lower()
        gender = "male" if gender == "m" else ("female" if gender == "f" else gender)
        return base, gender
    return s, None

def _build_meta(df: pd.DataFrame, fcfg: Dict[str, Any], base_meta: pd.DataFrame | None = None) -> pd.DataFrame:
    if base_meta is not None:
        return base_meta.copy()

    # case-insensitive domain lookup
    domain_by_stakeholder_raw = (fcfg.get("domain_by_stakeholder", {}) or {})
    domain_map = {_canon_key(k): _canon_text(v) for k, v in domain_by_stakeholder_raw.items()}

    rows = []
    unknown_stakeholders: set[str] = set()

    for col in _col_list(df):
        parts = [p.strip() for p in str(col).split("|")]
        if len(parts) >= 2:
            st_raw = parts[0]
            sg_raw = parts[1] if len(parts) >= 3 else ""
        else:
            st_raw, sg_raw = "", ""

        stakeholder = canon_stakeholder(st_raw)
        subgroup = canon_subgroup(sg_raw)

        dom = domain_map.get(_canon_key(st_raw), None)
        if dom is None:
            # default social if not explicitly mapped
            dom = "soc"
            if _canon_text(st_raw):
                unknown_stakeholders.add(_canon_text(st_raw))

        rows.append({
            "indicator": col,
            "domain": dom,
            "direction": "bad",
            "stakeholder": stakeholder,
            "subgroup": subgroup,
            "group_id": subgroup or stakeholder or dom,
        })

    if unknown_stakeholders:
        print("\nNOTE: Some stakeholder names in indicator columns were not found in domain_by_stakeholder; defaulted to 'soc'.")
        print("Examples:", ", ".join(sorted(list(unknown_stakeholders))[:12]) + ("..." if len(unknown_stakeholders) > 12 else ""))

    meta = pd.DataFrame(rows)

    if meta["indicator"].duplicated().any():
        d = meta.loc[meta["indicator"].duplicated(keep=False), "indicator"].value_counts().head(20)
        raise ValueError("Duplicate indicator names in meta. Examples:\n" + d.to_string())

    return meta

def _apply_meta_overrides(meta: pd.DataFrame, overrides: Dict[str, Any]) -> pd.DataFrame:
    meta2 = meta.copy()

    good_list = overrides.get("direction_good", []) or []
    if good_list:
        mask = meta2["indicator"].astype(str).isin(set(map(str, good_list)))
        meta2.loc[mask, "direction"] = "good"

    env_list = overrides.get("domain_env", []) or []
    if env_list:
        mask = meta2["indicator"].astype(str).isin(set(map(str, env_list)))
        meta2.loc[mask, "domain"] = "env"

    soc_list = overrides.get("domain_soc", []) or []
    if soc_list:
        mask = meta2["indicator"].astype(str).isin(set(map(str, soc_list)))
        meta2.loc[mask, "domain"] = "soc"

    return meta2

def _coalesce_gendered(df: pd.DataFrame, meta: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Gender rule:
      - If '<base> | total' exists: DROP male/female.
      - If total missing but male+female exist: CREATE total = mean(male,female), then DROP male/female.
      - If only male OR only female exists: keep as-is.
    """
    df2 = df.copy()
    meta2 = meta.copy()

    tmp = meta2.copy()
    tmp["base_indicator"], tmp["gender"] = zip(*tmp["indicator"].apply(_split_gender_suffix))

    created_totals = 0
    dropped_gender_cols = set()

    for (_stake, _subg, base), g in tmp.groupby(["stakeholder", "subgroup", "base_indicator"], dropna=False):
        gender_map = {}
        for _, r in g.iterrows():
            gen = r["gender"]
            if pd.isna(gen):
                continue
            gender_map[str(gen).lower()] = r["indicator"]

        male = gender_map.get("male")
        female = gender_map.get("female")
        total = gender_map.get("total")

        if total and (male or female):
            if male: dropped_gender_cols.add(male)
            if female: dropped_gender_cols.add(female)
            continue

        if (not total) and male and female:
            total_name = f"{base} | total"
            if total_name not in df2.columns:
                df2[total_name] = df2[[male, female]].mean(axis=1, skipna=True)
                base_row = g.iloc[0].to_dict()
                base_row["indicator"] = total_name
                meta2 = pd.concat([meta2, pd.DataFrame([base_row])], ignore_index=True)
                created_totals += 1

            dropped_gender_cols.add(male)
            dropped_gender_cols.add(female)

    if dropped_gender_cols:
        df2 = df2.drop(columns=[c for c in dropped_gender_cols if c in df2.columns], errors="ignore")
        meta2 = meta2.loc[~meta2["indicator"].isin(dropped_gender_cols)].reset_index(drop=True)

    return df2, meta2, {"created_totals": created_totals, "dropped_gender_cols": len(dropped_gender_cols)}

def _apply_column_filters(df: pd.DataFrame, meta: pd.DataFrame, fcfg: Dict[str, Any]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df2 = df.copy()
    meta2 = meta.copy()
    cols_cfg = fcfg.get("columns", {}) or {}

    whitelist = set(map(str, cols_cfg.get("whitelist_indicators", []) or []))

    # 1) exclude stakeholders (case-insensitive)
    ex_stake = set(_canon_key(x) for x in (cols_cfg.get("exclude_stakeholders", []) or []))
    if ex_stake:
        mask_drop = meta2["stakeholder"].apply(_canon_key).isin(ex_stake) & ~meta2["indicator"].astype(str).isin(whitelist)
        drop_inds = meta2.loc[mask_drop, "indicator"].tolist()
        df2 = df2.drop(columns=[c for c in drop_inds if c in df2.columns], errors="ignore")
        meta2 = meta2.loc[~mask_drop].reset_index(drop=True)

    # 2) exclude subgroup rules (case-insensitive)
    rules = cols_cfg.get("exclude_subgroups", []) or []
    if rules:
        mask_drop = pd.Series(False, index=meta2.index)
        for rule in rules:
            if not isinstance(rule, dict):
                continue
            st = _canon_key(rule.get("stakeholder", ""))
            sg = _canon_key(rule.get("subgroup", ""))
            hit = meta2["stakeholder"].apply(_canon_key).eq(st) & meta2["subgroup"].apply(_canon_key).eq(sg)
            mask_drop = mask_drop | hit
        mask_drop = mask_drop & ~meta2["indicator"].astype(str).isin(whitelist)
        drop_inds = meta2.loc[mask_drop, "indicator"].tolist()
        df2 = df2.drop(columns=[c for c in drop_inds if c in df2.columns], errors="ignore")
        meta2 = meta2.loc[~mask_drop].reset_index(drop=True)

    # 3) exclude explicit indicator names
    ex_inds = set(map(str, cols_cfg.get("exclude_indicators", []) or []))
    if ex_inds:
        mask_drop = meta2["indicator"].astype(str).isin(ex_inds) & ~meta2["indicator"].astype(str).isin(whitelist)
        drop_inds = meta2.loc[mask_drop, "indicator"].tolist()
        df2 = df2.drop(columns=[c for c in drop_inds if c in df2.columns], errors="ignore")
        meta2 = meta2.loc[~mask_drop].reset_index(drop=True)

    # 4) exclude by regex
    ex_regex = cols_cfg.get("exclude_regex", []) or []
    if ex_regex:
        pats = [re.compile(p) for p in ex_regex]
        def _hit(col: str) -> bool:
            return any(p.search(col) for p in pats)
        mask_drop = meta2["indicator"].astype(str).apply(_hit) & ~meta2["indicator"].astype(str).isin(whitelist)
        drop_inds = meta2.loc[mask_drop, "indicator"].tolist()
        df2 = df2.drop(columns=[c for c in drop_inds if c in df2.columns], errors="ignore")
        meta2 = meta2.loc[~mask_drop].reset_index(drop=True)

    # 5) gender coalescing
    df2, meta2, _ = _coalesce_gendered(df2, meta2)

    # 6) drop all-NaN / globally-constant indicators (keeps the matrix “well-posed”)
    ind_cols = _col_list(df2)
    drop = []
    for c in ind_cols:
        s = pd.to_numeric(df2[c], errors="coerce")
        if s.isna().all():
            drop.append(c); continue
        if s.nunique(dropna=True) <= 1:
            drop.append(c)
    if drop:
        df2 = df2.drop(columns=drop, errors="ignore")
        meta2 = meta2.loc[~meta2["indicator"].isin(drop)].reset_index(drop=True)

    return df2, meta2

def filters_apply(df: pd.DataFrame, filters_cfg: Dict[str, Any], base_meta: pd.DataFrame | None = None) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df2 = df.copy()

    rows_cfg = filters_cfg.get("rows", {}) or {}
    ex_c = set(map(str, rows_cfg.get("exclude_countries", []) or []))
    if ex_c:
        df2 = df2.loc[~df2["country"].astype(str).isin(ex_c)].reset_index(drop=True)

    ex_t = set(map(str, rows_cfg.get("exclude_techs", []) or []))
    if ex_t:
        df2 = df2.loc[~df2["tech"].astype(str).isin(ex_t)].reset_index(drop=True)

    meta = _build_meta(df2, filters_cfg, base_meta=base_meta)
    df3, meta3 = _apply_column_filters(df2, meta, filters_cfg)
    meta3 = _apply_meta_overrides(meta3, filters_cfg.get("meta_overrides", {}) or {})
    return df3, meta3

# =============================================================================
# 3) Missing-data guardrails (hard fail)
# =============================================================================
def assert_no_missing_indicators(df: pd.DataFrame, stage: str = "") -> None:
    keys = {"country", "tech"}
    ind_cols = [c for c in df.columns if c not in keys]

    X = df[ind_cols].apply(pd.to_numeric, errors="coerce")
    if X.isna().any().any():
        col_na = X.isna().sum(axis=0)
        col_na = col_na[col_na > 0].sort_values(ascending=False)
        bad_rows = X.isna().any(axis=1)
        examples = df.loc[bad_rows, ["country", "tech"]].head(15)
        raise ValueError(
            f"[{stage}] Missing or non-numeric indicator values detected.\n"
            f"Columns with missing values (count of NaNs):\n{col_na.to_string()}\n\n"
            f"Example affected (country, tech) rows:\n{examples.to_string(index=False)}\n\n"
            "Fix the upstream exports / raw CSVs so the score matrix is complete."
        )

# =============================================================================
# 4) Orientation & normalization (per-country min–max, with eps)
# =============================================================================
def apply_orientation(df: pd.DataFrame, meta: pd.DataFrame, profile: str = "EXP") -> pd.DataFrame:
    """
    Orient all indicators so that LOWER = BETTER.
    - Default direction: 'bad' (already lower-better).
    - If direction == 'good' for social indicators: flip sign.
    - Optional LIN: log10 for strictly-positive SOCIAL indicators (then orient).
    """
    out = df.copy()
    m = meta.set_index("indicator")
    keys = {"country", "tech"}
    ind_cols = [c for c in out.columns if c not in keys]

    for col in ind_cols:
        dom = m.at[col, "domain"] if col in m.index else "soc"
        direction = m.at[col, "direction"] if col in m.index else "bad"

        if profile.upper() == "LIN" and str(dom).strip().lower() != "env":
            s = pd.to_numeric(out[col], errors="coerce").astype(float)
            pos = s > 0
            s2 = s.copy()
            s2.loc[pos] = np.log10(s.loc[pos])
            out[col] = s2

        if str(dom).strip().lower() != "env" and direction == "good":
            out[col] = -pd.to_numeric(out[col], errors="coerce").astype(float)

    return out

def minmax_normalize_by_country(df: pd.DataFrame, eps: float = 1e-12) -> pd.DataFrame:
    """Min–max scale to [0,1] within each country. If range < eps -> set 0 (non-discriminating)."""
    out = df.copy()
    keys = {"country", "tech"}
    ind_cols = [c for c in out.columns if c not in keys]

    def _scale(g: pd.DataFrame) -> pd.DataFrame:
        X = g[ind_cols].apply(pd.to_numeric, errors="coerce").astype(float)
        mn = X.min(axis=0, skipna=True)
        mx = X.max(axis=0, skipna=True)
        rng = (mx - mn)

        Z = pd.DataFrame(0.0, index=X.index, columns=X.columns)
        ok = rng >= eps
        Z.loc[:, ok] = (X.loc[:, ok] - mn[ok]) / rng[ok]
        return Z

    out[ind_cols] = out.groupby("country", group_keys=False).apply(_scale)
    return out

# =============================================================================
# 5) Meta alignment to a concrete matrix X
# =============================================================================
def align_meta(meta: pd.DataFrame, X: pd.DataFrame) -> pd.DataFrame:
    """Keep only indicators present in X, and order meta to match X columns."""
    ind_cols = [c for c in X.columns if c not in {"country", "tech"}]
    if len(ind_cols) != len(set(ind_cols)):
        raise ValueError("Duplicate indicator columns detected in X.")
    m = meta[meta["indicator"].isin(ind_cols)].set_index("indicator").loc[ind_cols].reset_index()
    if len(m) != len(ind_cols):
        missing = [c for c in ind_cols if c not in set(meta["indicator"])]
        raise ValueError("Meta is missing indicators present in X: " + str(missing[:20]))
    return m

# =============================================================================
# 6) Weight sampling (stakeholder-focus only) + SMAA acceptabilities
# =============================================================================
@dataclass
class Draws:
    stakeholder_focus: Dict[str, np.ndarray]  # stakeholder -> (D, K)

def _stakeholder_seed(base_seed: int, stakeholder: str) -> int:
    h = zlib.adler32(stakeholder.encode("utf-8"))
    return int((base_seed + h) % (2**32 - 1))

def _dirichlet(rng: np.random.Generator, alpha: np.ndarray, size: int = 1) -> np.ndarray:
    alpha = np.asarray(alpha, dtype=float)
    if (alpha <= 0).any():
        raise ValueError("Dirichlet alphas must be > 0")
    return rng.dirichlet(alpha, size=size)

def _safe_equal_split(n: int) -> np.ndarray:
    return (np.ones((n,), float) / n) if n > 0 else np.zeros((0,), float)

def _indices_by_social_hierarchy(meta: pd.DataFrame) -> Dict[str, Dict[str, List[int]]]:
    """stakeholder -> subgroup -> list of indicator indices (K-index) for SOCIAL only."""
    social: Dict[str, Dict[str, List[int]]] = {}
    for k, row in enumerate(meta.itertuples(index=False)):
        dom = str(getattr(row, "domain", "")).strip().lower()
        if dom in {"env", "environment", "environmental"}:
            continue
        st = canon_stakeholder(getattr(row, "stakeholder", ""))
        sg = canon_subgroup(getattr(row, "subgroup", ""))
        social.setdefault(st, {}).setdefault(sg, []).append(k)
    return social

def sample(meta: pd.DataFrame, cfg: dict[str, Any]) -> Draws:
    s = cfg.get("smaa", {}) or {}
    D = int(s.get("n_draws", 50_000))
    base_seed = int(s.get("seed", 42))

    alpha_subgroup = float(s.get("alpha_subgroup", 1.0))
    alpha_indicators = float(s.get("alpha_indicators", 1.0))

    det_sg  = bool(s.get("deterministic_equal_subgroups", False))
    det_in  = bool(s.get("deterministic_equal_within_subgroup", False))

    social_h = _indices_by_social_hierarchy(meta)
    K = len(meta)

    if not social_h:
        raise ValueError("No social indicators found after filtering. Check config/inputs.")

    W_focus: Dict[str, np.ndarray] = {}

    for st in social_h.keys():
        rng = np.random.default_rng(_stakeholder_seed(base_seed, st))

        subgroups = list(social_h[st].keys())
        if len(subgroups) == 0:
            W_focus[st] = np.zeros((D, K), dtype=float)
            continue

        if det_sg:
            sub_w = np.broadcast_to(_safe_equal_split(len(subgroups)), (D, len(subgroups)))
        else:
            sub_w = _dirichlet(rng, np.full(len(subgroups), alpha_subgroup, float), size=D)

        W = np.zeros((D, K), dtype=float)

        for j, sg in enumerate(subgroups):
            idx = social_h[st][sg]
            m = len(idx)
            if m == 0:
                continue

            if det_in:
                w_in = np.broadcast_to(_safe_equal_split(m), (D, m))
            else:
                w_in = _dirichlet(rng, np.full(m, alpha_indicators, float), size=D)

            # stakeholder-focus => 100% of total weight distributed within this stakeholder
            W[:, idx] = (w_in.T * sub_w[:, j]).T

        row_sums = W.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        W_focus[st] = W / row_sums

    return Draws(stakeholder_focus=W_focus)

def acceptabilities(df_norm: pd.DataFrame, W: np.ndarray, block: int = 5000) -> pd.DataFrame:
    """Acceptability (probability of being best) per (country, tech). Lower is better."""
    keys = ["country", "tech"]
    ind_cols = [c for c in df_norm.columns if c not in keys]
    K = len(ind_cols)
    D = W.shape[0]
    if W.shape[1] != K:
        raise ValueError(f"W has K={W.shape[1]} but df_norm has K={K}")

    acc_rows = []
    for country, g in df_norm.groupby("country", sort=False):
        techs = g["tech"].to_numpy()
        G = g[ind_cols].to_numpy(dtype=float)  # (T, K)
        T = G.shape[0]
        if T == 0:
            continue

        counts = np.zeros(T, dtype=np.int64)

        for s in range(0, D, block):
            e = min(s + block, D)
            scores = G @ W[s:e].T
            winners = np.argmin(scores, axis=0)
            counts += np.bincount(winners, minlength=T)

        acc = counts / D
        for i, t in enumerate(techs):
            acc_rows.append({"country": country, "tech": t, "acceptability": float(acc[i])})

    return pd.DataFrame(acc_rows)

# =============================================================================
# 7) Plotting: 3×3 map grid (6 stakeholder-focus panels + 3 ReCiPe totals)
# =============================================================================
def _load_world_boundaries(world_file: Optional[Path], cache_dir: Path) -> gpd.GeoDataFrame:
    """
    Load Natural Earth admin_0 boundaries.
    Preference order:
      1) local file (world_file) if provided
      2) cached download in cache_dir
      3) download from official NACIS CDN, cache it, then read
    """
    if world_file is not None:
        return gpd.read_file(world_file)

    url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cached = cache_dir / "ne_110m_admin_0_countries.zip"
    if not cached.exists():
        print(f"Downloading basemap to cache: {cached}")
        urlretrieve(url, cached)

    return gpd.read_file(cached)

_WORLD_GDF = _load_world_boundaries(WORLD_FILE, WORLD_CACHE_DIR)

def _first_valid(world: gpd.GeoDataFrame, *cols: str) -> pd.Series:
    s = pd.Series(pd.NA, index=world.index, dtype="object")
    for c in cols:
        if c in world.columns:
            v = world[c].astype("object").replace("-99", pd.NA)
            s = s.combine_first(v)
    return s

_WORLD_GDF["ISO3_JOIN"] = _first_valid(
    _WORLD_GDF,
    "ISO_A3_EH",
    "ADM0_A3",
    "ISO_A3",
    "SOV_A3",
    "iso_a3",
)
if _WORLD_GDF["ISO3_JOIN"].isna().all():
    raise RuntimeError("Could not find a usable ISO3 column in the world layer.")
_WORLD_ISO3_COL = "ISO3_JOIN"

BASE_PALETTE: Dict[str, str] = {
    "Biomass gasification": "#2166AC",  # blue
    "Electrolysis":          "#1B9E77",  # teal-green
    "SMR (biomethane)":      "#E6AB02",  # bright gold
    "SMR (natural gas)":     "#D95F02",  # vivid orange
    "Coal gasification":     "#CE6DBD",  # pink-mauve
}

def extend_palette_for_used_labels(used_labels: List[str], base_palette: Dict[str, str]) -> Dict[str, str]:
    pal = dict(base_palette)
    fallback_colors = ["#666666", "#8c8c8c", "#4d4d4d", "#b3b3b3"]
    i = 0
    for lbl in sorted(set(used_labels)):
        if lbl not in pal and pd.notna(lbl):
            pal[lbl] = fallback_colors[i % len(fallback_colors)]
            i += 1
    return pal

RECIPE_TOTAL_ENDPOINTS = [
    "ReCiPe 2016 v1.03, endpoint (H) regionalized | total: ecosystem quality | ecosystem quality",
    "ReCiPe 2016 v1.03, endpoint (H) regionalized | total: human health | human health",
    "ReCiPe 2016 v1.03, endpoint (H) regionalized | total: natural resources | natural resources",
]

def _recipe_pretty_label(ind: str) -> str:
    if "ecosystem quality" in ind:
        return "ReCiPe total – ecosystem quality"
    if "human health" in ind:
        return "ReCiPe total – human health"
    if "natural resources" in ind:
        return "ReCiPe total – natural resources"
    return ind

def attach_clean_labels(df: pd.DataFrame, best_col: str = "best_tech") -> pd.DataFrame:
    out = df.copy()
    out["best_tech_canon"] = out[best_col].apply(normalize_tech_code)
    out["best_tech_clean"] = out["best_tech_canon"].map(TECH_LABELS).fillna(out[best_col].astype(str))
    return out

def winners_by_country(acc_df: pd.DataFrame, variant: str, regime: str) -> pd.DataFrame:
    g = acc_df[(acc_df["variant"] == variant) & (acc_df["regime"] == regime)]
    if g.empty:
        return pd.DataFrame(columns=["country", "best_tech", "best_acceptability"])
    idx = g.groupby("country")["acceptability"].idxmax()
    best = g.loc[idx, ["country", "tech", "acceptability"]].rename(
        columns={"tech": "best_tech", "acceptability": "best_acceptability"}
    )
    best = attach_clean_labels(best, best_col="best_tech")
    return best.sort_values("country").reset_index(drop=True)

def winners_by_country_recipe_total(df: pd.DataFrame, endpoint_col: str) -> pd.DataFrame:
    if endpoint_col not in df.columns:
        raise ValueError(f"Endpoint column {endpoint_col!r} not found.")
    tmp = df[["country", "tech", endpoint_col]].copy()
    tmp[endpoint_col] = pd.to_numeric(tmp[endpoint_col], errors="coerce")
    tmp = tmp.dropna(subset=[endpoint_col])
    if tmp.empty:
        return pd.DataFrame(columns=["country", "best_tech", "best_value"])
    idx = tmp.groupby("country")[endpoint_col].idxmin()
    best = tmp.loc[idx, ["country", "tech", endpoint_col]].rename(
        columns={"tech": "best_tech", endpoint_col: "best_value"}
    )
    best = attach_clean_labels(best, best_col="best_tech")
    return best.reset_index(drop=True)

def plot_stakeholder_and_recipe_maps_grid(
    acc_all: pd.DataFrame,
    df_recipe: pd.DataFrame,
    variant: str,
    regimes: Sequence[str],
    endpoints: Optional[list[str]] = None,
    title_suffix: str | None = None,
    tech_colors: Optional[Dict[str, str]] = None,
    dpi: int = 300,
    col_gap_mm: float = 1.0,
    row_gap_mm: float = 6.0,
    top_margin_mm: float = 20.0,
    bottom_margin_mm: float = 31.0,
    crop_south_lat: float = -60.0,
    savepath: Optional[Path | str] = None,
    acceptability_hatch_threshold: Optional[float] = 0.5,
    hatch_pattern: str = "//////",
) -> None:
    """
    3×3 panel:
      - Top 2 rows (6 panels): stakeholder-focus regimes (soc_focus:<Stakeholder>)
      - Bottom row (3 panels): ReCiPe endpoint totals (best tech by lowest total)

    Interpretation (intended for captions):
      - Stakeholder panels show the technology with highest acceptability (probability of being best under sampled weights).
      - ReCiPe panels show the technology with the lowest endpoint total.
      - Hatching marks ambiguous wins: best acceptability < acceptability_hatch_threshold.
    """
    regimes = list(regimes)
    if len(regimes) != 6:
        raise ValueError("Expected exactly 6 stakeholder regimes to fill the top 2 rows.")

    if endpoints is None:
        endpoints = RECIPE_TOTAL_ENDPOINTS
    endpoints = [e for e in endpoints if e in df_recipe.columns]
    if len(endpoints) != 3:
        raise ValueError("Expected 3 ReCiPe total endpoint columns in df_recipe (ecosystem/human health/resources).")

    def _nice_regime_label(reg: str) -> str:
        return reg.split("soc_focus:", 1)[1].strip() if reg.startswith("soc_focus:") else reg

    regime_best: Dict[str, pd.DataFrame] = {}
    used_labels: set[str] = set()
    world_codes = set(_WORLD_GDF[_WORLD_ISO3_COL].dropna().astype(str).str.upper().unique())

    # winners: stakeholder regimes
    for reg in regimes:
        best = winners_by_country(acc_all, variant=variant, regime=reg)
        if not best.empty:
            best = best.rename(columns={"country": "iso3"}).copy()
            best["iso3"] = best["iso3"].astype(str).str.upper()

            missing_iso3 = set(best["iso3"].dropna().unique()) - world_codes
            if missing_iso3:
                best = best.loc[~best["iso3"].isin(missing_iso3)].copy()
                print(
                    f"Note: {len(missing_iso3)} ISO3 codes are not present in the basemap and were not plotted "
                    f"({', '.join(sorted(list(missing_iso3))[:12])}{'...' if len(missing_iso3) > 12 else ''})."
                )

            used_labels.update(best["best_tech_clean"].dropna().unique())
        regime_best[reg] = best

    # winners: endpoints
    endpoint_best: Dict[str, pd.DataFrame] = {}
    for ep in endpoints:
        best = winners_by_country_recipe_total(df_recipe, ep)
        if not best.empty:
            best = best.rename(columns={"country": "iso3"}).copy()
            best["iso3"] = best["iso3"].astype(str).str.upper()

            missing_iso3 = set(best["iso3"].dropna().unique()) - world_codes
            if missing_iso3:
                best = best.loc[~best["iso3"].isin(missing_iso3)].copy()
                print(
                    f"Note: {len(missing_iso3)} ISO3 codes are not present in the basemap and were not plotted "
                    f"({', '.join(sorted(list(missing_iso3))[:12])}{'...' if len(missing_iso3) > 12 else ''})."
                )

            used_labels.update(best["best_tech_clean"].dropna().unique())
        endpoint_best[ep] = best

    palette = extend_palette_for_used_labels(
        sorted(used_labels),
        BASE_PALETTE if tech_colors is None else tech_colors
    )

    # bounds + crop
    minx_all, miny_all, maxx_all, maxy_all = _WORLD_GDF.total_bounds
    miny = max(miny_all, crop_south_lat)
    data_w = maxx_all - minx_all
    data_h = maxy_all - miny
    if data_w <= 0 or data_h <= 0:
        raise RuntimeError("World bounds invalid; cannot size the figure.")

    # geometry in inches
    fig_width_in = 174.0 / 25.4
    col_gap_in = col_gap_mm / 25.4
    row_gap_in = row_gap_mm / 25.4
    top_margin_in = top_margin_mm / 25.4
    bottom_margin_in = bottom_margin_mm / 25.4

    side_margin_mm = 1.0
    left_margin_in = side_margin_mm / 25.4
    right_margin_in = side_margin_mm / 25.4

    ncols, nrows = 3, 3
    usable_width_in = fig_width_in - left_margin_in - right_margin_in
    max_col_gap_in = usable_width_in / (ncols - 1) * 0.45
    if col_gap_in > max_col_gap_in:
        col_gap_in = max_col_gap_in

    map_w_in = (usable_width_in - (ncols - 1) * col_gap_in) / ncols
    map_h_in = map_w_in * (data_h / data_w)
    fig_height_in = top_margin_in + bottom_margin_in + nrows * map_h_in + (nrows - 1) * row_gap_in

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(fig_width_in, fig_height_in), dpi=dpi)
    axes = np.asarray(axes).reshape(-1)

    labels = list(regimes) + list(endpoints)

    for i, label in enumerate(labels):
        ax = axes[i]

        if i < 6:
            best = regime_best[label]
            title_str = _nice_regime_label(label)
        else:
            best = endpoint_best[label]
            title_str = _recipe_pretty_label(label)

        if best.empty:
            gdf = _WORLD_GDF.assign(fill_color="#f0f0f0")
            gdf.plot(ax=ax, color=gdf["fill_color"], edgecolor="black", linewidth=0.2, aspect=None)
        else:
            gdf = _WORLD_GDF.merge(best, left_on=_WORLD_ISO3_COL, right_on="iso3", how="left")

            def _col(lbl):
                if pd.isna(lbl):
                    return "#f0f0f0"
                return palette.get(lbl, "#999999")

            gdf["fill_color"] = gdf["best_tech_clean"].apply(_col)

            # base fill
            gdf.plot(ax=ax, color=gdf["fill_color"], edgecolor="black", linewidth=0.2, aspect=None)

            # uncertainty cue: hatch if best_acceptability < threshold (stakeholder panels only)
            if (i < 6) and (acceptability_hatch_threshold is not None) and ("best_acceptability" in gdf.columns):
                m = pd.to_numeric(gdf["best_acceptability"], errors="coerce")
                low = m.notna() & (m < float(acceptability_hatch_threshold)) & gdf["best_tech_clean"].notna()
                if low.any():
                    gdf.loc[low].plot(
                        ax=ax,
                        facecolor="none",
                        edgecolor="black",
                        linewidth=0.0,
                        hatch=hatch_pattern,
                        aspect=None,
                    )

        ax.set_axis_off()
        ax.set_xlim(minx_all, maxx_all)
        ax.set_ylim(miny, maxy_all)
        ax.set_title(title_str, fontsize=9)

        ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False,
                               edgecolor="black", linewidth=0.4, clip_on=False))

    title_base = "Hydrogen technology by country – highest acceptability (stakeholder focus) and lowest ReCiPe endpoint total"
    title_text = f"{title_base} ({title_suffix})" if title_suffix else title_base
    suptitle_y = 1.0 - (top_margin_in / (2.0 * fig_height_in))
    #fig.suptitle(title_text, fontsize=9, y=suptitle_y)

    # Legend order: prefer canonical tech order
    preferred_order = [TECH_LABELS[k] for k in TECH_LABELS.keys()]
    ordered_used = [t for t in preferred_order if t in used_labels] + [t for t in sorted(used_labels) if t not in preferred_order]

    handles: List[Patch] = [
        Patch(facecolor=palette[t], edgecolor="black", linewidth=0.3, label=t)
        for t in ordered_used
    ]

    # no "No data" legend entry by design
    if acceptability_hatch_threshold is not None:
        handles.append(
            Patch(
                facecolor="#ffffff",
                edgecolor="black",
                linewidth=0.3,
                hatch=hatch_pattern,
                label=f"Low acceptability (<{acceptability_hatch_threshold:.2f})",
            )
        )

    legend_center_y = bottom_margin_in / (2.0 * fig_height_in)
    leg = fig.legend(
        handles=handles, title="Technology",
        loc="lower center", bbox_to_anchor=(0.5, legend_center_y),
        fontsize=7, title_fontsize=7,
        ncol=min(3, len(handles)),
        frameon=True, columnspacing=1.0, handlelength=1.5, handletextpad=0.4,
    )
    leg.get_frame().set_linewidth(0.4)
    leg.get_frame().set_edgecolor("black")

    left_frac = left_margin_in / fig_width_in
    right_frac = 1.0 - (right_margin_in / fig_width_in)
    top_frac = 1.0 - (top_margin_in / fig_height_in)
    bottom_frac = bottom_margin_in / fig_height_in

    wspace = col_gap_in / map_w_in
    hspace = row_gap_in / map_h_in
    fig.subplots_adjust(left=left_frac, right=right_frac, top=top_frac, bottom=bottom_frac, wspace=wspace, hspace=hspace)

    if savepath:
        savepath = Path(savepath)
        savepath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(savepath, dpi=dpi, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

# =============================================================================
# 8) Robustness demo (second figure) — UPDATED LABELS + FILTERING
# =============================================================================
def score_margins_for_country(df_norm: pd.DataFrame, W: np.ndarray, country: str) -> np.ndarray:
    """Return margin=(2nd best score - best score) per draw for one country; lower score is better."""
    keys = ["country", "tech"]
    ind_cols = [c for c in df_norm.columns if c not in keys]
    g = df_norm[df_norm["country"] == country].copy()
    G = g[ind_cols].to_numpy(float)          # (T,K)
    scores = G @ W.T                         # (T,D)
    s_sorted = np.sort(scores, axis=0)       # ascending: best then 2nd best
    if s_sorted.shape[0] < 2:
        return np.full((scores.shape[1],), np.nan)
    return (s_sorted[1, :] - s_sorted[0, :]) # larger => clearer separation

def _fig2_palette() -> Dict[str, str]:
    """Palette keyed by *clean technology labels* (same as Figure 1 maps)."""
    used_labels = [TECH_LABELS[k] for k in TECH_LABELS.keys()]
    return extend_palette_for_used_labels(used_labels, BASE_PALETTE)

def robustness_demo(
    acc_all: pd.DataFrame,
    df_norm: pd.DataFrame,
    draws: Draws,
    stakeholder: str,
    variant: str,
    tech_order: Optional[Sequence[str]] = None,
    savepath: Optional[Path | str] = None,
):
    """
    Figure 2:
      - Top row: acceptability bars for a "strong" and "weak" country (all technologies shown).
      - Bottom row: histograms of score margins (2nd-best − best) across weight draws.
    """
    stakeholder = canon_stakeholder(stakeholder)
    reg = f"soc_focus:{stakeholder}"

    g = acc_all[(acc_all["variant"] == variant) & (acc_all["regime"] == reg)].copy()
    if g.empty:
        raise ValueError(f"No rows for {reg} and variant={variant}")

    # pick strong/weak countries based on best acceptability in each country
    win = g.loc[g.groupby("country")["acceptability"].idxmax()].copy()
    win = win.sort_values("acceptability", ascending=False)

    strong_country = win.iloc[0]["country"]
    weak_country   = win.iloc[-1]["country"]

    W = draws.stakeholder_focus[stakeholder]

    # consistent tech order + labels + colors (match Figure 1)
    if tech_order is None:
        tech_order = list(TECH_LABELS.keys())  # canonical order from your mapping
    tech_order = [normalize_tech_code(t) for t in tech_order]

    tech_labels = [TECH_LABELS.get(t, str(t)) for t in tech_order]
    pal = _fig2_palette()
    tech_colors = [pal.get(lbl, "#999999") for lbl in tech_labels]

    def _bar(ax, country: str):
        gg = g[g["country"] == country][["tech", "acceptability"]].copy()
        gg["tech"] = gg["tech"].apply(normalize_tech_code)

        # map acceptabilities onto the full tech order (missing -> 0)
        acc_map = dict(zip(gg["tech"].astype(str), gg["acceptability"].astype(float)))
        heights = [float(acc_map.get(t, 0.0)) for t in tech_order]

        # winner annotation (within this country)
        best_i = int(np.argmax(heights)) if len(heights) else 0
        best_lbl = tech_labels[best_i] if tech_labels else ""
        best_acc = heights[best_i] if heights else float("nan")

        ax.bar(
            tech_labels,
            heights,
            color=tech_colors,
            edgecolor="black",
            linewidth=0.3,
        )
        ax.set_ylim(0, 1)
        ax.set_ylabel("Acceptability", fontsize=8)
        ax.tick_params(axis="x", labelrotation=25, labelsize=7)
        ax.tick_params(axis="y", labelsize=7)

        # subtle y-grid improves readability without clutter
        ax.yaxis.grid(True, linewidth=0.3, alpha=0.4)
        ax.set_axisbelow(True)

        ax.set_title(
            f"{country} – winner: {best_lbl} (acc={best_acc:.2f})",
            fontsize=9
        )

        # optionally highlight very ambiguous outcomes with a light note
        if best_acc < 0.5:
            ax.text(
                0.99, 0.92,
                "ambiguous winner",
                transform=ax.transAxes,
                ha="right", va="top",
                fontsize=7
            )

    # margins
    m_strong = score_margins_for_country(df_norm, W, strong_country)
    m_weak   = score_margins_for_country(df_norm, W, weak_country)

    # shared x-limits for comparability (robust to outliers)
    m_all = np.concatenate([m_strong[np.isfinite(m_strong)], m_weak[np.isfinite(m_weak)]])
    if m_all.size:
        x_max = float(np.quantile(m_all, 0.99))
        if x_max <= 0:
            x_max = float(np.max(m_all))
    else:
        x_max = 1.0

    _rob_w_in = 174.0 / 25.4
    fig = plt.figure(figsize=(_rob_w_in, _rob_w_in * 6.2 / 11), dpi=220)

    ax1 = plt.subplot(2, 2, 1)
    _bar(ax1, strong_country)
    ax1.set_title(f"{ax1.get_title()}\n({reg})", fontsize=9)

    ax2 = plt.subplot(2, 2, 2)
    _bar(ax2, weak_country)
    ax2.set_title(f"{ax2.get_title()}\n({reg})", fontsize=9)

    ax3 = plt.subplot(2, 2, 3)
    ax3.hist(m_strong[np.isfinite(m_strong)], bins=40)
    ax3.set_title(f"{strong_country} – margin distribution", fontsize=9)
    ax3.set_xlabel("margin (2nd-best − best)", fontsize=8)
    ax3.set_ylabel("count", fontsize=8)
    ax3.tick_params(labelsize=7)
    ax3.set_xlim(0, x_max)

    ax4 = plt.subplot(2, 2, 4)
    ax4.hist(m_weak[np.isfinite(m_weak)], bins=40)
    ax4.set_title(f"{weak_country} – margin distribution", fontsize=9)
    ax4.set_xlabel("margin (2nd-best − best)", fontsize=8)
    ax4.set_ylabel("count", fontsize=8)
    ax4.tick_params(labelsize=7)
    ax4.set_xlim(0, x_max)

    plt.tight_layout()
    if savepath:
        savepath = Path(savepath)
        savepath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(savepath, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

# =============================================================================
# 9) RUN PIPELINE
# =============================================================================
def write_manifest(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

# 9.1 Build raw matrix from node CSVs
df_raw = build_score_matrix(RAW_DIR)
df_raw = standardize_country_to_iso3(df_raw, col="country")
df_raw = canonicalize_techs_inplace(df_raw, col="tech")
display(df_raw.head())

# 9.1b Coverage check: do all countries have all technologies?
EXPECTED_TECHS = list(TECH_LABELS.keys())
report_tech_coverage(df_raw[["country", "tech"]], expected_techs=EXPECTED_TECHS)

# 9.2 Apply filters + build meta
df_filt, meta = filters_apply(df_raw, CFG["filters"])
print("After filters:", df_filt.shape, "| meta rows:", len(meta))

# 9.2b (Informative) report if any ReCiPe totals are missing by column
present_endpoints = [c for c in RECIPE_TOTAL_ENDPOINTS if c in df_raw.columns]
if len(present_endpoints) == 3:
    na_counts = df_raw[present_endpoints].isna().sum().sort_values(ascending=False)
    if na_counts.max() > 0:
        print("\n=== ReCiPe total endpoint missingness (raw) ===")
        display(na_counts.to_frame("NaN_count"))
else:
    recipe_like = [c for c in df_raw.columns if "ReCiPe 2016 v1.03, endpoint" in str(c)]
    print("\nWARNING: Could not find all expected ReCiPe total endpoint columns.")
    print("Found columns containing 'ReCiPe 2016 v1.03, endpoint':")
    print("\n".join(recipe_like[:50]))

# 9.3 Hard fail if any missing indicators (your requirement)
assert_no_missing_indicators(df_filt, stage="after filters")

# 9.4 Orientation + normalization (per country)
df_oriented = apply_orientation(df_filt, meta, profile=VARIANT)
assert_no_missing_indicators(df_oriented, stage="after orientation")

df_norm = minmax_normalize_by_country(df_oriented, eps=NORM_EPS)
assert_no_missing_indicators(df_norm, stage="after normalization")

# 9.5 Align meta to normalized matrix
meta_aligned = align_meta(meta, df_norm)

# 9.6 Sample stakeholder-focus weights
draws = sample(meta_aligned, CFG)

# 9.7 Compute acceptabilities for the 6 stakeholder panels
CANON_STAKEHOLDERS = [canon_stakeholder(s) for s in [
    "Children",
    "Consumer",
    "Local Community",
    "Society",
    "Value Chain Actors",
    "Workers",
]]
available = sorted(draws.stakeholder_focus.keys())
missing = [s for s in CANON_STAKEHOLDERS if s not in draws.stakeholder_focus]
if missing:
    raise ValueError(
        "Required stakeholder groups are missing from draws.stakeholder_focus:\n  - "
        + "\n  - ".join(missing)
        + "\n\nAvailable stakeholders are:\n  - "
        + "\n  - ".join(available)
    )

acc_tables = []
for st in CANON_STAKEHOLDERS:
    Wmat = draws.stakeholder_focus[st]
    acc = acceptabilities(df_norm, Wmat, block=5000)
    acc["variant"] = VARIANT
    acc["regime"] = f"soc_focus:{st}"
    acc_tables.append(acc)

acc_all = pd.concat(acc_tables, ignore_index=True)
print("acc_all rows:", len(acc_all))
print("Available regimes:", sorted(acc_all["regime"].unique()))

# 9.8 Build ReCiPe totals df_recipe from df_raw (restricted to tech/country pairs used)
pairs_used = acc_all[["country", "tech"]].drop_duplicates()

present_endpoints = [c for c in RECIPE_TOTAL_ENDPOINTS if c in df_raw.columns]
if len(present_endpoints) != 3:
    recipe_like = [c for c in df_raw.columns if "ReCiPe 2016 v1.03, endpoint" in str(c)]
    raise ValueError(
        "Could not find the expected 3 ReCiPe total endpoint columns in df_raw.\n"
        "Expected (exact match):\n  - " + "\n  - ".join(RECIPE_TOTAL_ENDPOINTS) + "\n\n"
        f"Found {len(present_endpoints)} of them.\n"
        "Columns containing 'ReCiPe 2016 v1.03, endpoint' in df_raw:\n  - "
        + "\n  - ".join(recipe_like[:50])
    )

df_recipe = df_raw[["country", "tech"] + present_endpoints].copy()
df_recipe = pairs_used.merge(df_recipe, on=["country", "tech"], how="left")

print("df_recipe rows:", len(df_recipe), "| countries:", df_recipe["country"].nunique())

miss_by_ep = df_recipe[present_endpoints].isna().sum().sort_values(ascending=False)
if miss_by_ep.max() > 0:
    print("\n=== ReCiPe total endpoint missingness (used pairs) ===")
    display(miss_by_ep.to_frame("NaN_count"))
    ex = df_recipe.loc[df_recipe[present_endpoints].isna().any(axis=1), ["country", "tech"] + present_endpoints].head(20)
    print("Example missing rows:")
    display(ex)

missing_all_three = df_recipe[present_endpoints].isna().all(axis=1).mean()
print(f"Share of (country, tech) pairs with all endpoint totals missing: {missing_all_three:.1%}")

# --- FIGURE 1: 3×3 grid (6 stakeholders + ReCiPe totals) ---
plot_stakeholder_and_recipe_maps_grid(
    acc_all=acc_all,
    regimes=[f"soc_focus:{s}" for s in CANON_STAKEHOLDERS],
    df_recipe=df_recipe,
    variant=VARIANT,
    title_suffix=VARIANT,
    dpi=DPI,
    col_gap_mm=COL_GAP_MM,
    row_gap_mm=ROW_GAP_MM,
    top_margin_mm=TOP_MARGIN_MM,
    bottom_margin_mm=BOTTOM_MARGIN_MM,
    crop_south_lat=CROP_SOUTH_LAT,
    savepath=FIG_SAVE_PATH,
    acceptability_hatch_threshold=ACCEPTABILITY_HATCH_THRESHOLD,
    hatch_pattern="//////",
)

print("Saved figure to:", FIG_SAVE_PATH.resolve() if FIG_SAVE_PATH else "(not saved)")

# --- FIGURE 2: robustness demo (one stakeholder) ---
robustness_demo(acc_all, df_norm, draws, stakeholder="Society", variant=VARIANT, savepath=FIG2_SAVE_PATH)
print("Saved figure to:", FIG2_SAVE_PATH.resolve() if FIG2_SAVE_PATH else "(not saved)")

# --- Optional: write a small manifest (good for SI / reproducibility) ---
if MANIFEST_PATH is not None:
    payload = {
        "timestamp_local": datetime.now().isoformat(timespec="seconds"),
        "timestamp_local": datetime.now().isoformat(timespec="seconds"),
        "raw_dir": str(RAW_DIR),
        "variant": VARIANT,
        "n_draws": int(CFG["smaa"]["n_draws"]),
        "seed": int(CFG["smaa"]["seed"]),
        "n_countries": int(df_raw["country"].nunique()),
        "n_techs": int(df_raw["tech"].nunique()),
        "expected_techs": EXPECTED_TECHS,
        "n_indicators_after_filters": int(len([c for c in df_filt.columns if c not in {"country","tech"}])),
        "acceptability_hatch_threshold": float(ACCEPTABILITY_HATCH_THRESHOLD),
        "recipe_endpoints_present": present_endpoints,
        "recipe_missing_any_share_pairs_used": float(df_recipe[present_endpoints].isna().any(axis=1).mean()),
        "recipe_missing_all_share_pairs_used": float(missing_all_three),
        "world_source": str(WORLD_FILE) if WORLD_FILE else f"cached:{WORLD_CACHE_DIR / 'ne_110m_admin_0_countries.zip'}",
    }
    write_manifest(MANIFEST_PATH, payload)
    print("Wrote manifest to:", MANIFEST_PATH.resolve())
